![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 4 -- Lab 1: Full ML Pipeline

A hospital has collected clinical measurements from 800 patients and wants to predict which patients are at risk of **diabetes**. The raw data is messy — some values are missing, some features are text, scales differ wildly, and the classes are imbalanced.

Build a **complete machine-learning pipeline from scratch**:

| Stage | Goal |
|---|---|
| EDA | Understand the data before touching it |
| Preprocessing | Make it model-ready |
| Splitting | Stratified 80/20 |
| Modeling | Logistic regression + L2 regularization in NumPy |
| Training | Gradient-descent loops with loss tracking |
| Evaluation | Confusion matrix, P/R/F1, decision boundary |

**Dataset:** `diabetes_patients.csv` — 800 records, 8 features + 1 binary target


---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')

---
# Part 1 — Exploratory Data Analysis

## Task 1: Load & Inspect

Load the CSV. Figure out what you're working with — shapes, types, basic stats. Which columns are numerical? Which are categorical?

In [ ]:
# Your code here

## Task 2: Missing Values

Find and visualize where data is missing. How much is missing in each column?

In [ ]:
# Your code here

## Task 3: Class Distribution

How balanced is the target variable? Visualize and report the ratio. Think about what a naive "always predict majority class" model would score.

In [ ]:
# Your code here

## Task 4: Outliers & Correlations

Use box plots to spot outliers in the numerical columns. Build a correlation heatmap to see which features relate to the target and to each other.

In [ ]:
# Your code here

---
# Part 2 — Data Preprocessing

## Task 5: Impute Missing Values

You can't feed `NaN` to a model. Fill numerical gaps with the **median** and categorical gaps with the **mode**. Confirm nothing is left missing.

In [ ]:
# Your code here

## Task 6: Remove Outliers

Apply the **IQR method** to `bmi` and `blood_glucose_level`. Remove rows that fall outside $[Q_1 - 1.5 \cdot \text{IQR},\; Q_3 + 1.5 \cdot \text{IQR}]$. Report how many rows were dropped.

In [ ]:
# Your code here

## Task 7: Encode Categorical Features

Convert the string columns to numbers using **one-hot encoding** (drop the first category to avoid multicollinearity).

In [ ]:
# Your code here

## Task 8: Feature Scaling

Standardize every feature to zero mean and unit variance **using NumPy** (no sklearn):

$$z = \frac{x - \mu}{\sigma}$$

Separate `X` (features) and `y` (target) before scaling. Keep the feature names around — you'll need them later.

In [ ]:
# Your code here

---
# Part 3 — Data Splitting

## Task 9: Stratified Train / Test Split

Split 80/20 using sklearn's `train_test_split` with `stratify=y`. Print the class percentages in each split to verify the ratio is preserved.

In [ ]:
# Your code here

---
# Part 4 — Modeling (From Scratch)

## Task 10: Sigmoid & Binary Cross-Entropy

The **sigmoid** squashes any real number into $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Binary cross-entropy** measures the gap between predicted probabilities and true labels:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\right]$$

Implement both. Watch out for numerical edge cases (`log(0)`, overflow in `exp`).

In [ ]:
# Your code here

## Task 11: Gradient Computation

For logistic regression with predictions $\mathbf{p} = \sigma(\mathbf{X}\mathbf{w} + b)$, the BCE gradients are:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{n}\mathbf{X}^{\top}(\mathbf{p} - \mathbf{y})$$

$$\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)$$

Implement a function that takes `(X, y, w, b)` and returns `(dw, db)`.

In [ ]:
# Your code here

## Task 12: L2 Regularization

Add a **Ridge penalty** to discourage large weights:

$$\mathcal{J} = \mathcal{L} + \frac{\lambda}{2n}\sum_{j=1}^{d} w_j^2$$

The gradient picks up an extra term (the bias $b$ is **not** regularized):

$$\frac{\partial \mathcal{J}}{\partial \mathbf{w}} = \frac{1}{n}\mathbf{X}^{\top}(\mathbf{p} - \mathbf{y}) + \frac{\lambda}{n}\mathbf{w}$$

Implement the regularized loss and gradient. Sanity-check: with $\lambda = 0$ the results should match Task 11.

In [ ]:
# Your code here

---
# Part 5 — Training & Testing Loops

## Task 13: Train Two Models

Follow this blueprint for a single training run:

```
w ← zeros   b ← 0

for each epoch:
    p       ← σ(Xw + b)            # forward
    loss    ← loss_fn(y, p, w, …)   # measure
    dw, db  ← grad_fn(X, y, w, b, …)# backward
    w       ← w − α · dw            # update
    b       ← b − α · db

    record train_loss
    compute & record test_loss       # NO update on test!
```

Train **two models** — one without regularization (Model A) and one with $\lambda = 1.0$ (Model B). Use `learning_rate = 0.1`, `epochs = 500` for both. Store all weights and loss histories.

In [ ]:
# Your code here — Model A (no regularization)

In [ ]:
# Your code here — Model B (L2 regularized, λ = 1.0)

## Task 14: Loss Curves

Plot train loss vs. test loss for both models side by side. Look for:
- **Overfitting**: train loss keeps dropping but test loss diverges.
- **Underfitting**: both losses stay high.
- **Good fit**: both losses converge to similar values.

In a comment, state what you observe and which model generalizes better.

In [ ]:
# Your code here

---
# Part 6 — Evaluation

Pick the better model from Part 5 and evaluate it on the test set.

In [ ]:
# Your code here — assign w_best, b_best

## Task 15: Confusion Matrix

Predict on the test set (threshold = 0.5). Compute **TP, FP, FN, TN** using NumPy and visualize the confusion matrix as a heatmap.

In [ ]:
# Your code here

## Task 16: Precision, Recall & F1

Compute all three **from scratch** (no sklearn) using your TP / FP / FN values:

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN} \qquad F_1 = \frac{2 \cdot P \cdot R}{P + R}$$

Also compute accuracy. In a comment, explain why accuracy alone is misleading here.

In [ ]:
# Your code here

## Task 17: Decision Boundary

Pick the **2 features most correlated with the target**, retrain a 2-feature logistic regression (same loop, $\lambda = 1.0$, 500 epochs), and plot the test data colored by true class with the decision boundary overlaid.

For a 2-feature model $z = w_1 x_1 + w_2 x_2 + b$, the boundary is the line where $z = 0$:

$$x_2 = -\frac{w_1}{w_2}\, x_1 - \frac{b}{w_2}$$


In [ ]:
# Your code here

## Task 18: Final Analysis

Answer in the cell below:

1. Which model generalizes better and how do the loss curves show it?
2. Where exactly do you see overfitting?
3. A naive "always predict 0" model gets ~70% accuracy. Is your model actually learning? Which metric proves it?


*Your answers here*

---
## Conclusion

You built a complete ML pipeline from raw CSV to evaluated model:

| Stage | What You Did |
|---|---|
| **EDA** | Inspected data, found missing values, checked class balance, spotted outliers |
| **Preprocessing** | Imputed NaNs, removed outliers, one-hot encoded strings, standardized features |
| **Splitting** | Stratified 80/20 split preserving class ratio |
| **Modeling** | Logistic regression + L2 regularization from scratch in NumPy |
| **Training** | Gradient descent with train/test loss tracking |
| **Evaluation** | Confusion matrix, P/R/F1, decision boundary |
